# Step 7 — Parametric surrogate across the PFC phase diagram

The biggest limitation today is that the model is **non-parametric and single-phase**:
one frozen regime, so everything looks the same. This notebook trains a
**parametric** FNO conditioned on (r, n0) over a phase-spanning dataset, so a
*single model* covers **stripe / hexagonal / liquid** — and then shows the payoff:

> the parametric FNO **reconstructs the PFC phase diagram** in one forward pass
> per point, agreeing with the solver but orders of magnitude faster.

This is the step that turns the tool from a demo into a parametric research instrument.
Set **Runtime → T4 GPU**, then **Run all**.

### 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/Tntguy0128/crystal-growth-sim.git
%cd crystal-growth-sim
!pip install -q torch numpy scipy matplotlib pyyaml tensorboard
import sys; sys.path.insert(0,'crystal_tool')

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU, then Run all."
print("GPU OK:", torch.cuda.get_device_name(0))

### 2. Phase-spanning dataset
Neutral random-noise seeds swept across (r, n0) so the equilibrium PHASE is set
by the parameters, not a planted lattice. Reuses the Drive zip if present.

In [ ]:
import os
z='/content/drive/MyDrive/pfc_dataset_phases.zip'
if os.path.exists(z):
    !cp "$z" .; unzip -o -q pfc_dataset_phases.zip
else:
    !python generate_dataset.py --sweep --config pfc_config_phases.yaml
    !zip -r -q pfc_dataset_phases.zip data_pfc_phases; cp pfc_dataset_phases.zip /content/drive/MyDrive/
# phase distribution check
import glob, numpy as np, analyze as A
from collections import Counter
tally=Counter()
for f in glob.glob('data_pfc_phases/*.npz'):
    zz=np.load(f,allow_pickle=True); tally[A.classify_phase(zz['n_all'][-1],float(zz['dx']))]+=1
print(len(glob.glob('data_pfc_phases/*.npz')),'trajectories | phases:',dict(tally))

### 3. Train the parametric FNO (more rigorous than the quick ablations)
`include_conditioning: true` appends r, n0 as input channels. Longer training
(more epochs / patience) than the earlier 40-epoch runs, per the next-steps plan.

In [ ]:
import yaml, subprocess, shutil, os
with open('config.yaml') as f: cfg=yaml.safe_load(f)
cfg['data']['data_dir']='data_pfc_phases'
cfg['data']['split_by']='config'
cfg['data']['include_conditioning']=True       # <-- parametric: condition on (r, n0)
cfg['data']['normalize_conditioning']=True
cfg['train']['physics_weight']=0.0
cfg['train']['pde_weight']=0.0
cfg['train']['epochs']=120                      # more rigorous training
cfg['train']['patience']=30
out_dir='runs/parametric_phases'
cfg['logging']['out_dir']=out_dir; cfg['eval']['out_dir']=f'{out_dir}/eval'
with open('config.yaml','w') as f: yaml.dump(cfg,f,default_flow_style=False,sort_keys=False)
env={**os.environ,'PYTHONUNBUFFERED':'1'}
print('===== TRAIN parametric (conditioned on r, n0) =====',flush=True)
subprocess.run(['python','-u','train_fno.py','--config','config.yaml'],check=True,env=env)
shutil.copy(f'{out_dir}/best.pt','/content/drive/MyDrive/fno_parametric_phases.pt')
print('saved -> Drive/fno_parametric_phases.pt',flush=True)

### 4. Standard rollout evaluation

In [ ]:
import subprocess, os, pandas as pd
env={**os.environ,'PYTHONUNBUFFERED':'1'}
subprocess.run(['python','-u','evaluate_fno.py','--config','config.yaml','--checkpoint',f'{out_dir}/best.pt'],check=True,env=env)
df=pd.read_csv(f'{out_dir}/eval/per_trajectory.csv')
print('mean rollout MSE:',round(float(df['rollout_mse'].mean()),5))

### 5. The payoff — the parametric FNO reconstructs the phase diagram
For each (r, n0) we grow a crystal from neutral noise with the **single** trained
model (conditioned on r, n0) and classify the resulting phase. We compare to the
PFC solver and to the wall-clock time.

In [ ]:
import numpy as np, time, sys
sys.path.insert(0,'crystal_tool')
import grow, analyze
from pfc_solver import PFCConfig, PFCSolver, make_initial_condition
import matplotlib.pyplot as plt

model, ck, device = grow.load_fno(f'{out_dir}/best.pt')
R=[-0.20,-0.30,-0.40]; N0=[0.0,-0.10,-0.18,-0.25,-0.32,-0.40]
COL={'hexagonal':'#0E9594','stripe':'#E0913A','liquid':'#9FB0C0'}

def noise_field(r,n0,seed=3):
    cfg=PFCConfig(r=r,n0=n0,seed_type='random_noise',noise_amplitude=0.10,T=250.0,rng_seed=seed)
    return make_initial_condition(cfg,np.random.default_rng(seed)).astype('float32'), cfg

t0=time.time(); fno_ph={}
for r in R:
    for n0 in N0:
        f0,cfg=noise_field(r,n0)
        fr=grow.grow_fno(model,ck,device,f0,steps=40,cfg=cfg)   # conditioned rollout
        fno_ph[(r,n0)]=analyze.classify_phase(fr[-1],cfg.dx)
t_fno=time.time()-t0

t0=time.time(); sol_ph={}
for r in R:
    for n0 in N0:
        res=PFCSolver(PFCConfig(r=r,n0=n0,seed_type='random_noise',noise_amplitude=0.10,T=250.0,rng_seed=3)).run()
        sol_ph[(r,n0)]=analyze.classify_phase(res.n_all[-1],float(16*np.pi/128))
t_sol=time.time()-t0

agree=sum(fno_ph[k]==sol_ph[k] for k in fno_ph)/len(fno_ph)
print(f'phase agreement FNO vs solver: {agree*100:.0f}%')
print(f'wall time: FNO {t_fno:.1f}s  vs  solver {t_sol:.1f}s  ->  {t_sol/max(t_fno,1e-6):.0f}x faster')

fig,ax=plt.subplots(1,2,figsize=(13,5))
for a,(ph,title) in zip(ax,[(sol_ph,'PFC solver (ground truth)'),(fno_ph,'Parametric FNO (one model)')]):
    for (r,n0),p in ph.items():
        a.scatter(n0,r,s=520,marker='s',color=COL.get(p,'#000'),edgecolors='w',linewidths=1.5)
    a.set_xlabel('mean density n0'); a.set_ylabel('quench depth r'); a.set_title(title); a.grid(alpha=0.2)
import matplotlib.lines as ml
ax[1].legend(handles=[ml.Line2D([0],[0],marker='s',color='w',markerfacecolor=c,markersize=12,label=p) for p,c in COL.items()],loc='upper left',fontsize=9)
fig.suptitle(f'Parametric FNO reconstructs the phase diagram — {agree*100:.0f}% match, {t_sol/max(t_fno,1e-6):.0f}x faster',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.savefig('runs/parametric_phase_diagram.png',dpi=110); plt.show()
!cp runs/parametric_phase_diagram.png /content/drive/MyDrive/

### Takeaway
A single conditioned model spans the phase diagram and reproduces it far faster
than the solver. This is the foundation for the parametric studies the
phase-field literature targets — and it gives the UI real diversity: moving the
r / n0 sliders now changes the **phase**, not just the orientation.
Point Crystal Studio at `fno_parametric_phases.pt` to drive it live.